# v0.14.4 — Record-id lookups

A record's `id` column holds a native `surrealdb.RecordID`, not a string. Until v0.14.4 the
`WHERE` builder bound whatever Python value you passed, so `filter(id="alice")` compiled to
`id = $_f0` with a plain string — and SurrealDB does not consider a `RecordID` equal to the
string form of its identifier. The lookup matched **nothing**, with no error and no warning
([issue #159](https://github.com/EulogySnowfall/SurrealDB-ORM-lite/issues/159)).

This notebook walks what v0.14.4 changed:

1. the forms `filter(id=...)` accepts
2. why `int` and `str` ids are different records
3. `get()` on a non-string id
4. `get_or_create(id=...)` converging instead of duplicating
5. text lookups on `id` failing loudly

Everything here behaves **identically on SurrealDB 2.6.x and 3.x** — the fix is in how the ORM
compiles the query, not in any server-version-specific feature.

In [1]:
import os

from surreal_orm_lite import SurrealDBConnectionManager

HOST = os.environ.get("SURREALDB_HOST", "localhost")
PORT = os.environ.get("SURREALDB_PORT", "8000")
SurrealDBConnectionManager.set_connection(
    url=f"ws://{HOST}:{PORT}/rpc",
    user="root",
    password="root",
    namespace="examples",
    database="examples",
)
print("Connection configured:", SurrealDBConnectionManager.is_connection_set())

Connection configured: True


## 1. Models

Two models that differ only in how their id is typed — that difference is the whole point of
section 2.

In [2]:
from surrealdb import RecordID

from surreal_orm_lite import BaseSurrealModel


class Member(BaseSurrealModel):
    id: str | RecordID | None = None
    name: str
    age: int = 0


class Ticket(BaseSurrealModel):
    id: int | RecordID | None = None
    label: str


# Start from a clean slate.
client = await SurrealDBConnectionManager.get_client()
for table in ("Member", "Ticket"):
    try:
        await client.query(f"REMOVE TABLE {table};")
    except Exception:
        pass

await Member(id="alice", name="Alice", age=30).save()
await Member(id="bob", name="Bob", age=25).save()
await Ticket(id=5, label="five").save()

rows = await client.query("SELECT * FROM Member;")
print("stored id type:", type(rows[0]["id"]).__name__, "->", rows[0]["id"])

stored id type: RecordID -> Member:alice


## 2. The forms `filter(id=...)` accepts

All four address the same record. The full `"Member:alice"` form is unwrapped only when the
prefix names the table being queried — `"Other:alice"` may genuinely *be* the stored string
id, so it is left alone.

In [3]:
by_bare = await Member.objects().filter(id="alice").exec()
by_thing = await Member.objects().filter(id="Member:alice").exec()
by_record_id = await Member.objects().filter(id=RecordID("Member", "alice")).exec()
by_in = await Member.objects().filter(id__in=["alice", "bob"]).exec()

print("bare identifier :", [m.name for m in by_bare])
print("table:id form   :", [m.name for m in by_thing])
print("RecordID        :", [m.name for m in by_record_id])
print("id__in          :", sorted(m.name for m in by_in))

bare identifier : ['Alice']
table:id form   : ['Alice']
RecordID        : ['Alice']
id__in          : ['Alice', 'Bob']


## 3. `int` and `str` ids are different records

The value's **Python type** decides which record is addressed — the same rule `save()` uses
when it writes the row. `Ticket` declares `id: int`, so its record id is the *integer* `5`;
the string `"5"` would be a different record that was never created.

In [4]:
print("filter(id=5)   ->", [t.label for t in await Ticket.objects().filter(id=5).exec()])
print('filter(id="5") ->', [t.label for t in await Ticket.objects().filter(id="5").exec()])

filter(id=5)   -> ['five']
filter(id="5") -> []


## 4. `get()` follows the same rule

`get()` used to build `RecordID(table, str(id_item))`, so an integer id was unreachable
through it as well. It now shares the helper with `filter()`.

In [5]:
print("get(5)   ->", (await Ticket.objects().get(5)).label)

try:
    await Ticket.objects().get("5")
except Exception as exc:
    print('get("5") ->', type(exc).__name__, "— the string record id was never created")

get(5)   -> five
get("5") -> SurrealDbNotFoundError — the string record id was never created


## 5. `get_or_create(id=...)` converges

This is the failure that motivated the issue. The lookup never matched, so **every** call took
the create path — and the second one hit `already exists`. Run this cell twice: the count stays
at one.

In [6]:
first, created_first = await Member.objects().get_or_create(defaults={"name": "Carol"}, id="carol")
second, created_second = await Member.objects().get_or_create(defaults={"name": "Carol"}, id="carol")

print("first call  created:", created_first)
print("second call created:", created_second)
print("rows named Carol   :", len(await Member.objects().filter(name="Carol").exec()))

first call  created: True
second call created: False
rows named Carol   : 1


## 6. Text lookups on `id` fail loudly

A record id is not a string, so `contains` / `startswith` / `regex` / `like` and friends could
never match. Returning an empty set made that indistinguishable from "no such record"; they now
raise, and the message points at the raw-query escape hatch.

In [7]:
try:
    await Member.objects().filter(id__startswith="al").exec()
except ValueError as exc:
    print("ValueError:", exc)

# The escape hatch, if you really want the textual form of the id:
rows = await client.query(
    "SELECT * FROM Member WHERE string::starts_with(record::id(id), 'al');"
)
print("via raw query:", [row["name"] for row in rows])

ValueError: 'startswith' cannot be applied to the record id column 'id': a record id is not a string. Filter on a regular column, or reach for the textual form explicitly in a raw query (e.g. string::contains(record::id(id), '...')).
via raw query: ['Alice']


## 7. An aliased primary key is untouched

A model that aliases its identity through `primary_key` stores that alias as an **ordinary
column** next to the RecordID `id`. It is a plain string comparison, so it always worked and is
deliberately left alone — only the column literally named `id` is coerced.

In [8]:
class Product(BaseSurrealModel):
    model_config = {"primary_key": "code"}

    code: str
    label: str


try:
    await client.query("REMOVE TABLE Product;")
except Exception:
    pass

await Product(code="abc", label="Widget").save()

row = (await client.query("SELECT * FROM Product;"))[0]
print("stored row:", {k: str(v) for k, v in row.items()})
print("filter(code='abc') ->", [p.label for p in await Product.objects().filter(code="abc").exec()])
print("filter(id='abc')   ->", [p.label for p in await Product.objects().filter(id="abc").exec()])

stored row: {'code': 'abc', 'id': 'Product:abc', 'label': 'Widget'}
filter(code='abc') -> ['Widget']
filter(id='abc')   -> ['Widget']


## 8. Cleanup

In [9]:
for table in ("Member", "Ticket", "Product"):
    try:
        await client.query(f"REMOVE TABLE {table};")
    except Exception:
        pass

await SurrealDBConnectionManager.close_connection()
print("Cleaned up.")

Cleaned up.
